In [ ]:
import numpy as np
import pandas as pd

homes = pd.read_csv("../data/homes_clean.csv", parse_dates=["sold"])
homes["area"] = homes["area"].str.title()
homes["months"] = (homes["sold"].dt.year - 2021) * 12 + homes["sold"].dt.month - 1

In [ ]:
features = ["sqft", "beds", "built", "garage", "condition", "area", "months"]
X = homes[features]
y = homes["price"]
X.shape, y.shape

In [ ]:
X.head(3)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=1460
)
len(X_train), len(X_test)

In [ ]:
y_train.median(), y_test.median()

In [ ]:
from sklearn.linear_model import LinearRegression

def log_features(X, y):
    return pd.DataFrame({
        "log_sqft": np.log(X["sqft"]),
        "log_per_sqft": np.log(y / X["sqft"]),
    })

leaky = LinearRegression().fit(log_features(X_train, y_train), np.log(y_train))
round(leaky.score(log_features(X_test, y_test), np.log(y_test)), 4)

In [ ]:
honest = LinearRegression().fit(np.log(X_train[["sqft"]]), np.log(y_train))
round(honest.score(np.log(X_test[["sqft"]]), np.log(y_test)), 4)

In [ ]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False).fit(X_train[["area"]])
encoder.get_feature_names_out()

In [ ]:
pd.DataFrame(
    encoder.transform(X_train[["area"]].head(3)),
    columns=encoder.get_feature_names_out(),
    index=X_train["area"].head(3),
)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().fit(X_train[["sqft", "built"]])
scaler.mean_.round(1), scaler.scale_.round(1)

In [ ]:
scaled = scaler.transform(X_test[["sqft", "built"]])
pd.DataFrame(scaled, columns=["sqft", "built"]).agg(["mean", "std"]).round(3)